# Simple RAG (Retrieval-Augmented Generation) System

In [1]:
import os
import sys
import re
from dotenv import load_dotenv
from langchain_groq import ChatGroq
# PDF Loader
from langchain_community.document_loaders import PyMuPDFLoader
# Text Splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
# FAISS
from langchain_community.vectorstores import FAISS

from langchain_huggingface import HuggingFaceEmbeddings

from langchain_core.prompts import ChatPromptTemplate


load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
HUGGINGFACE_API_KEY = os.getenv("HUGGINGFACE_API_KEY")


C:\Users\sk335\AppData\Local\Temp\ipykernel_25144\2349267085.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader
c:\Users\sk335\OneDrive\Documents\Coding\PythonProjects\rag_techniques\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Environment variables

In [2]:
if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY is not set")

if not HUGGINGFACE_API_KEY:
    raise ValueError("HUGGINGFACE_API_KEY is not set")

GROQ_MODEL = os.getenv(
    "GROQ_MODEL",
    "llama-3.1-8b-instant"
)

EMBEDDING_MODEL = os.getenv(
    "EMBEDDING_MODEL",
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Groq model:", GROQ_MODEL)
print("Embedding model:", EMBEDDING_MODEL)

Groq model: openai/gpt-oss-20b
Embedding model: sentence-transformers/all-MiniLM-L6-v2


# PDF path

In [3]:
PDF_PATH = "../data/Sandeep.pdf"

# Text Cleaning

In [4]:
def replace_t_with_space(text: str) -> str:
    """
    Clean Common PDF extraction whitespace problem. 
    """

    text = text.replace("\t", " ")

    # Multiple space -> single space
    text = re.sub(r"[\t]+", " ", text)

    # Too many new lines -> two new lines
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

# Huggingface embeddings

In [5]:
def get_embedding():
    return HuggingFaceEmbeddings(
        model_name=EMBEDDING_MODEL,
        model_kwargs = {
            "device": "cpu"
        },

        encode_kwargs = {
            "normalize_embeddings": True
        }
    )

# Load PDF

In [6]:
def load_pdf(pdf_path: str):
    loader = PyMuPDFLoader(pdf_path)
    documents = loader.load()
    print(f"Loaded pages: {len(documents)}")

    return documents

In [7]:
documents = load_pdf(PDF_PATH)

print(documents[0].page_content[:1000])

Loaded pages: 1
Sandeep Singh
Full Stack Developer
# sandeep.necoder@gmail.com
 Twitter
 Portfolio
§ GitHub
Professional Summary
Full Stack Developer experienced in building scalable web applications using the MERN stack, Next.js, and
FastAPI. Skilled in developing secure REST APIs, implementing JWT authentication, and crafting responsive UIs
with Tailwind CSS. Proficient in working with MongoDB, Cloudinary, and familiar with AI/LLM tools like
LangChain and Gemini API.
Technical Skills
Languages: JavaScript, TypeScript, Python
Frontend: HTML5, CSS3, React.js, Next.js, Redux Toolkit, React Query, TanStack Query, Tailwind CSS
Backend: Node.js, Express.js, FastAPI, GraphQL, LangChain, Gemini API, REST APIs, JWT Auth, OAuth
Database: MongoDB Atlas, Prisma, PostgreSQL, ChromaDB
Tools: Git, Vite, Clerk, Postman, Docker, Cloudinary, ImageKit
Projects
AI Career Coach – AI-Powered Career Platform  Live Project — § GitHub Repository
Tech Stack: Next.js, JavaScript, Prisma, PostgreSQL, Clerk, 

# Split PDF into chunks

In [8]:
def split_documents(documents, chunk_size=1000, chunk_overlap=150):
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap, separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ])

    chunks = splitter.split_documents(documents)

    for chunk in chunks:
        chunk.page_content = replace_t_with_space(chunk.page_content)
    
    print(f"Total chunks: {len(chunks)}")

    return chunks

In [9]:
chunks = split_documents(documents)
print(chunks[0].page_content)

Total chunks: 4
Sandeep Singh
Full Stack Developer
# sandeep.necoder@gmail.com
 Twitter
 Portfolio
§ GitHub
Professional Summary
Full Stack Developer experienced in building scalable web applications using the MERN stack, Next.js, and
FastAPI. Skilled in developing secure REST APIs, implementing JWT authentication, and crafting responsive UIs
with Tailwind CSS. Proficient in working with MongoDB, Cloudinary, and familiar with AI/LLM tools like
LangChain and Gemini API.
Technical Skills
Languages: JavaScript, TypeScript, Python
Frontend: HTML5, CSS3, React.js, Next.js, Redux Toolkit, React Query, TanStack Query, Tailwind CSS
Backend: Node.js, Express.js, FastAPI, GraphQL, LangChain, Gemini API, REST APIs, JWT Auth, OAuth
Database: MongoDB Atlas, Prisma, PostgreSQL, ChromaDB
Tools: Git, Vite, Clerk, Postman, Docker, Cloudinary, ImageKit
Projects
AI Career Coach – AI-Powered Career Platform  Live Project — § GitHub Repository


# Create FAISS vector store

In [10]:
def create_vector_store(chunks):
    embeddings = get_embedding()

    vector_store = FAISS.from_documents(
        documents=chunks,
        embedding=embeddings
    )

    return vector_store

In [11]:
vector_store = create_vector_store(chunks)

print("FAISS vector store created successfully.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10900.89it/s]


FAISS vector store created successfully.


# Create retriever

In [12]:
TOP_K = 2

retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": TOP_K
    }
)

In [13]:
question = "What is the main cause of climate change?"

retrieved_docs = retriever.invoke(question)

print(
    f"Retrieved {len(retrieved_docs)} documents"
)

Retrieved 2 documents


In [14]:
def show_context(docs):

    for i, doc in enumerate(docs, start=1):

        page = doc.metadata.get(
            "page",
            "unknown"
        )

        print(
            f"\n========== Context {i} | Page {page} =========="
        )

        print(
            doc.page_content
        )

In [15]:
show_context(retrieved_docs)


========== Context 1 | Page 0 ==========
Sandeep Singh
Full Stack Developer
# sandeep.necoder@gmail.com
 Twitter
 Portfolio
§ GitHub
Professional Summary
Full Stack Developer experienced in building scalable web applications using the MERN stack, Next.js, and
FastAPI. Skilled in developing secure REST APIs, implementing JWT authentication, and crafting responsive UIs
with Tailwind CSS. Proficient in working with MongoDB, Cloudinary, and familiar with AI/LLM tools like
LangChain and Gemini API.
Technical Skills
Languages: JavaScript, TypeScript, Python
Frontend: HTML5, CSS3, React.js, Next.js, Redux Toolkit, React Query, TanStack Query, Tailwind CSS
Backend: Node.js, Express.js, FastAPI, GraphQL, LangChain, Gemini API, REST APIs, JWT Auth, OAuth
Database: MongoDB Atlas, Prisma, PostgreSQL, ChromaDB
Tools: Git, Vite, Clerk, Postman, Docker, Cloudinary, ImageKit
Projects
AI Career Coach – AI-Powered Career Platform  Live Project — § GitHub Repository

========== Context 2 | Page 0 ===

# Initialize Groq

In [16]:
llm = ChatGroq(
    model=GROQ_MODEL,
    temperature=0,
    max_tokens=1024,
    api_key=GROQ_API_KEY
)

print("Groq LLM initialized.")

Groq LLM initialized.


# RAG prompt

In [17]:
RAG_PROMPT = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You are a helpful RAG assistant.

Answer the user's question using ONLY the
provided context.

Rules:

1. Do not invent information.
2. If the answer is not present in the context,
   say that the document does not contain enough
   information.
3. Keep the answer clear and concise.
4. Use the document context as the primary source.
"""
        ),

        (
            "human",
            """
Context:

{context}

Question:

{question}

Answer:
"""
        )
    ]
)

# Format retrieved documents

In [18]:
def format_docs(docs):

    return "\n\n".join(
        [
            f"""
[Page {doc.metadata.get("page", "unknown")}]

{doc.page_content}
"""
            for doc in docs
        ]
    )

# Complete RAG function

In [19]:
def ask_rag(
    question: str,
    k: int = 2
):

    # Retrieve documents
    docs = vector_store.as_retriever(
        search_type="similarity",
        search_kwargs={
            "k": k
        }
    ).invoke(question)

    # Convert documents into context
    context = format_docs(docs)

    # Create prompt
    messages = RAG_PROMPT.invoke(
        {
            "context": context,
            "question": question
        }
    )

    # Call Groq
    response = llm.invoke(messages)

    return response.content, docs

## Ask question

In [26]:
question = "project name "

answer, docs = ask_rag(question)

print("QUESTION:")
print(question)

print("\nANSWER:")
print(answer)

QUESTION:
project name 

ANSWER:
**Project names mentioned in the context:**

- AI Career Coach – AI‑Powered Career Platform  
- ShopWiz – Scalable E‑commerce Platform


In [23]:
print("\nSOURCES:")

for i, doc in enumerate(docs, start=1):

    print(
        f"{i}. Page {doc.metadata.get('page', 'unknown')}"
    )


SOURCES:
1. Page 0
2. Page 0
